# Alpha Zero General C++ - Google Colab GPU Training

This notebook sets up the Alpha Zero training pipeline with GPU acceleration on Google Colab (T4 GPU).

In [ ]:
# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory
checkpoint_dir = '/content/drive/MyDrive/alpha_zero_checkpoints/'
os.makedirs(checkpoint_dir, exist_ok=True)

In [ ]:
# Install dependencies
!pip install torch torchvision pybind11 coloredlogs tqdm

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone or navigate to the project
%cd /content
# If you have the project in Drive, copy it here
# Or set up your project path
PROJECT_PATH = '/content/alpha-zero-general-cpp'
os.chdir(PROJECT_PATH)

In [ ]:
# Compile C++ extension
!python setup.py build_ext --inplace

In [ ]:
# Test GPU neural network
import sys
import numpy as np
from othello.OthelloGame import OthelloGame
from othello.pytorch.NNetGPU import NNetGPUWrapper

game = OthelloGame(6)
nnet = NNetGPUWrapper(game)

# Test prediction
board = np.zeros((6, 6))
board[2, 2] = 1
board[3, 3] = -1
pi, v = nnet.predict(board)
print(f"Prediction test: pi shape={pi.shape}, value={v}")

# Test batch prediction
boards = np.random.randn(32, 6, 6).astype(np.float64)
pis, vs = nnet.predict_batch(boards)
print(f"Batch prediction: pis shape={pis.shape}, vs shape={vs.shape}")

In [ ]:
# Run training
import logging
import coloredlogs
from Coach import Coach
from utils import dotdict

coloredlogs.install(level='INFO')

args = dotdict({
    'numIters': 10,
    'numEps': 25,
    'tempThreshold': 15,
    'updateThreshold': 0.6,
    'maxlenOfQueue': 200000,
    'numMCTSSims': 25,
    'arenaCompare': 10,
    'cpuct': 1,
    'checkpoint': checkpoint_dir,
    'load_model': False,
    'load_folder_file': (checkpoint_dir, 'best.pth.tar'),
    'numItersForTrainExamplesHistory': 20,
})

g = Game(6)
nnet = NNetGPUWrapper(g)
c = Coach(g, nnet, args)
c.learn()

In [ ]:
# Quick evaluation / pit against best model
from pit import Arena

args_eval = dotdict({
    'arenaCompare': 20,
    'numMCTSSims': 50,
    'cpuct': 1,
})

arena = Arena(g, nnet, nnet, args_eval)
arena.playGames(20)